# 🛒 Retail Sales Performance Analysis
**Dataset:** Sample Superstore — 9,994 orders across 4 US regions  
**Goal:** Identify revenue trends, profit drivers, and underperforming segments  
**Tools:** Python, Pandas, Matplotlib, Seaborn

## Cell 1 — Load & Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('SampleSuperstore.csv', encoding='latin-1')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head())
print('\nNulls:', df.isnull().sum().sum())
print('Regions:', df['Region'].unique())
print('Categories:', df['Category'].unique())

## Cell 2 — Business Overview

In [ ]:
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
profit_margin = (total_profit / total_sales) * 100
total_orders  = len(df)

print('=' * 40)
print(f'  Total Sales:     ${total_sales:>12,.2f}')
print(f'  Total Profit:    ${total_profit:>12,.2f}')
print(f'  Profit Margin:   {profit_margin:>11.2f}%')
print(f'  Total Orders:    {total_orders:>12,}')
print('=' * 40)

## Cell 3 — Sales & Profit by Region and Category

In [ ]:
region = df.groupby('Region')[['Sales','Profit']].sum().reset_index().sort_values('Sales', ascending=False)
cat    = df.groupby('Category')[['Sales','Profit']].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Sales & Profit by Region and Category', fontsize=14, fontweight='bold')

x = np.arange(len(region))
axes[0].bar(x - 0.2, region['Sales'],  0.4, label='Sales',  color='#1A5276')
axes[0].bar(x + 0.2, region['Profit'], 0.4, label='Profit', color='#AED6F1')
axes[0].set_xticks(x)
axes[0].set_xticklabels(region['Region'])
axes[0].set_title('Sales & Profit by Region')
axes[0].set_ylabel('Amount ($)')
axes[0].legend()

colors = ['#E74C3C' if p < 0 else '#1A5276' for p in cat['Profit']]
axes[1].bar(cat['Category'], cat['Profit'], color=colors)
axes[1].set_title('Profit by Category (Red = Loss)')
axes[1].set_ylabel('Profit ($)')
for i, v in enumerate(cat['Profit']):
    axes[1].text(i, v + 100, f'${v:,.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('region_category.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 4 — Segment Analysis & Discount Impact

In [ ]:
seg = df.groupby('Segment')[['Sales','Profit']].sum().reset_index().sort_values('Sales', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Segment Performance & Discount Impact', fontsize=14, fontweight='bold')

x = np.arange(len(seg))
axes[0].bar(x - 0.2, seg['Sales'],  0.4, label='Sales',  color='#1A5276')
axes[0].bar(x + 0.2, seg['Profit'], 0.4, label='Profit', color='#AED6F1')
axes[0].set_xticks(x)
axes[0].set_xticklabels(seg['Segment'])
axes[0].set_title('Sales & Profit by Customer Segment')
axes[0].set_ylabel('Amount ($)')
axes[0].legend()

axes[1].scatter(df['Discount'], df['Profit'], alpha=0.3, color='#1A5276', s=10)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Discount vs Profit\n(High discounts drive losses)')
axes[1].set_xlabel('Discount Rate')
axes[1].set_ylabel('Profit ($)')

plt.tight_layout()
plt.savefig('segment_discount.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 5 — Profit Heatmap: Region vs Category

In [ ]:
pivot = df.groupby(['Region','Category'])['Profit'].sum().reset_index()
pivot_table = pivot.pivot(index='Region', columns='Category', values='Profit')

plt.figure(figsize=(9, 5))
sns.heatmap(pivot_table, annot=True, fmt='.0f', cmap='RdYlGn', linewidths=0.5, annot_kws={'size': 11})
plt.title('Profit Heatmap: Region vs Category\n(Red = Loss, Green = Profit)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('profit_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

loss = pivot[pivot['Profit'] < 0]
print('Loss-making segments:')
print(loss)

## Cell 6 — Sub-Category Profit Analysis

In [ ]:
sub = df.groupby('Sub-Category')['Profit'].sum().reset_index().sort_values('Profit')
colors = ['#E74C3C' if p < 0 else '#1A5276' for p in sub['Profit']]

plt.figure(figsize=(10, 7))
plt.barh(sub['Sub-Category'], sub['Profit'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Profit by Sub-Category (Red = Loss)', fontsize=13, fontweight='bold')
plt.xlabel('Profit ($)')
plt.tight_layout()
plt.savefig('subcategory_profit.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 3 loss-making sub-categories:')
print(sub.head(3)[['Sub-Category','Profit']].to_string(index=False))

## ✅ Key Findings
- **Total Sales:** $2,297,200 across 9,994 orders with an overall **profit margin of 12.47%**
- **West region** generates the highest profit; **Central region** is the weakest
- **Furniture category generates a net loss in the Central region** ($-2,871) despite high sales volume
- **Tables** ($-17,725) and **Bookcases** ($-3,472) are the biggest loss-making sub-categories
- **High discounts (>40%) consistently lead to negative profit** — discount strategy needs review
- **Consumer segment** drives the most revenue; **Corporate segment** has better profit margins

**Business Recommendation:** Discontinue or reprice Tables and Bookcases in the Central region. Cap discounts at 30% to protect profit margins.

## Cell 7 — SQL Analysis using SQLite
> Same dataset queried using SQL for structured business reporting

In [ ]:
import sqlite3

# Load dataframe into SQLite in-memory database
conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, if_exists='replace', index=False)
print('Data loaded into SQLite successfully!')
print(f'Total rows: {len(df)}')

In [ ]:
# Query 1: Sales & Profit by Region with Margin
query1 = '''
    SELECT Region,
           ROUND(SUM(Sales), 2)  AS Total_Sales,
           ROUND(SUM(Profit), 2) AS Total_Profit,
           ROUND(SUM(Profit)*100.0/SUM(Sales), 2) AS Profit_Margin_Pct
    FROM orders
    GROUP BY Region
    ORDER BY Total_Sales DESC
'''
print('Sales & Profit by Region:')
display(pd.read_sql_query(query1, conn))

In [ ]:
# Query 2: Loss-making Region + Category combos
query2 = '''
    SELECT Region, Category,
           ROUND(SUM(Profit), 2) AS Total_Profit
    FROM orders
    GROUP BY Region, Category
    HAVING SUM(Profit) < 0
    ORDER BY Total_Profit ASC
'''
print('Loss-making Region + Category combinations:')
display(pd.read_sql_query(query2, conn))

In [ ]:
# Query 3: Top 5 loss-making Sub-Categories
query3 = '''
    SELECT [Sub-Category],
           ROUND(SUM(Profit), 2) AS Total_Profit
    FROM orders
    GROUP BY [Sub-Category]
    ORDER BY Total_Profit ASC
    LIMIT 5
'''
print('Top 5 Loss-making Sub-Categories:')
display(pd.read_sql_query(query3, conn))

In [ ]:
# Query 4: Discount impact on average profit
query4 = '''
    SELECT
        CASE
            WHEN Discount = 0       THEN '0% Discount'
            WHEN Discount <= 0.2    THEN '1-20% Discount'
            WHEN Discount <= 0.4    THEN '21-40% Discount'
            ELSE '40%+ Discount'
        END AS Discount_Band,
        COUNT(*)            AS Total_Orders,
        ROUND(AVG(Profit), 2) AS Avg_Profit
    FROM orders
    GROUP BY Discount_Band
    ORDER BY Avg_Profit DESC
'''
print('Impact of Discount on Average Profit:')
display(pd.read_sql_query(query4, conn))
conn.close()